In [12]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul3-train/1_3_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
x_train = x_train.astype(np.float32)
print(x_train.shape, x_train.dtype, y_train.shape, y_train.dtype)

test_data = np.load('Numarul3-test/1_3_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
x_test = x_test.astype(np.float32)
print(x_test.shape, x_test.dtype, y_test.shape, y_test.dtype)

n_classes = len(np.unique(ar=y_train))
print(f"n_classes = {n_classes}")

x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)

mean = torch.mean(input=x_train, dim=0)
std = torch.std(input=x_train, dim=0)

x_train_norm = (x_train - mean) / std
x_test_norm = (x_test - mean) / std

n = x_train_norm.shape[0]
K = (x_train_norm.T @ x_train_norm) / (n - 1)
print(K)

D, V = torch.linalg.eigh(K)
idx = torch.argsort(input=D, descending=True)
D = D[idx]
V = V[:, idx]

x_train = x_train_norm @ V[:, :1024]
x_test = x_test_norm @ V[:, :1024]

(25000, 1024) float32 (25000,) int64
(5000, 1024) float32 (5000,) int64
n_classes = 5
tensor([[1.0000, 0.9742, 0.9417,  ..., 0.2786, 0.2949, 0.3180],
        [0.9742, 1.0000, 0.9768,  ..., 0.2721, 0.2873, 0.3036],
        [0.9417, 0.9768, 1.0000,  ..., 0.2669, 0.2806, 0.2948],
        ...,
        [0.2786, 0.2721, 0.2669,  ..., 1.0000, 0.9702, 0.9221],
        [0.2949, 0.2873, 0.2806,  ..., 0.9702, 1.0000, 0.9665],
        [0.3180, 0.3036, 0.2948,  ..., 0.9221, 0.9665, 1.0000]])


In [13]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.feature_vectors = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature_vector = self.feature_vectors[idx]
        label = self.labels[idx]

        feature_vector = torch.as_tensor(data=feature_vector, dtype=torch.float32)
        label = torch.as_tensor(data=label, dtype=torch.int64)
        
        return feature_vector, label

train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [14]:
class Model(nn.Module):
    def __init__(self, in_feautures=1024, n_classes=n_classes):
        super().__init__()
        self.act1 = nn.functional.relu
        self.act2 = nn.functional.sigmoid
        self.fc1 = nn.Linear(in_features=in_feautures, out_features=512) 
        self.fc2 = nn.Linear(in_features=512, out_features=in_feautures) 
        self.fc3 = nn.Linear(in_features=in_feautures, out_features=256) 
        self.fc4 = nn.Linear(in_features=256, out_features=128) 
        self.fc5 = nn.Linear(in_features=128, out_features=32)
        self.fc6 = nn.Linear(in_features=32, out_features=n_classes)  

    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        x = self.act1(self.fc3(x))
        x = self.act1(self.fc4(x))
        x = self.act2(self.fc5(x))
        x = self.act1(self.fc6(x))

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 15

def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total

best = 0
for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"train_loss: {train_loss:.4f} || train_acc: {train_acc:.4f} || test_loss: {test_loss:.4f} || test_acc: {test_acc:.4f}")
    if best < test_acc:
        best = test_acc
        torch.save(model, 'model.pt')

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')

  7%|▋         | 1/15 [00:01<00:15,  1.10s/it]

train_loss: 1.3867 || train_acc: 0.4135 || test_loss: 1.2878 || test_acc: 0.4684


 13%|█▎        | 2/15 [00:02<00:14,  1.12s/it]

train_loss: 1.1811 || train_acc: 0.5336 || test_loss: 1.1852 || test_acc: 0.5330


 20%|██        | 3/15 [00:03<00:13,  1.13s/it]

train_loss: 1.0020 || train_acc: 0.6182 || test_loss: 1.1546 || test_acc: 0.5458


... (20 outputs left out)

 93%|█████████▎| 14/15 [00:15<00:01,  1.07s/it]

train_loss: 0.1442 || train_acc: 0.9631 || test_loss: 1.8114 || test_acc: 0.5616


100%|██████████| 15/15 [00:16<00:00,  1.11s/it]

train_loss: 0.1333 || train_acc: 0.9664 || test_loss: 1.8458 || test_acc: 0.5574
0.5738
